# COCO Dataset Exploration: Indian Traffic Images, Categories, and Bounding-Box Annotations
**Course Outcome 1 (CO1 - Skilling Session 5)**: Deep Learning with Accelerated Computing (25CS3112E)
- **Explore COCO JSON Structure**: `images`, `categories`, `annotations`
- **Filter Indian Traffic Categories**: `person`, `bicycle`, `car`, `motorcycle`, `bus`, `truck`, `traffic light`, `stop sign`, `auto_rickshaw`
- **Coordinate Transformation**: Convert COCO `[x, y, width, height]` to `[x1, y1, x2, y2]`
- **Visual Ground Truth Inspection**: Render Indian traffic scenes with class-specific bounding boxes
- **Dataset Audit**: Analyze class distribution and identify class imbalance


In [ ]:
# Cell 1: Environment Setup & Clone Week - 2 Components
!git clone https://github.com/pavankumar7744-droid/DeepLearning_Skills.git
%cd /content/DeepLearning_Skills/week\ -\ 2/
!pip install -q matplotlib seaborn pandas ultralytics
print('Week-5 Environment Initialized Successfully.')


In [ ]:
# Cell 2: COCO JSON Structure Definition & Traffic Categories Mapping
coco_traffic_categories = [
    {"id": 1, "name": "person", "supercategory": "person"},
    {"id": 2, "name": "bicycle", "supercategory": "vehicle"},
    {"id": 3, "name": "car", "supercategory": "vehicle"},
    {"id": 4, "name": "motorcycle", "supercategory": "vehicle"},
    {"id": 6, "name": "bus", "supercategory": "vehicle"},
    {"id": 8, "name": "truck", "supercategory": "vehicle"},
    {"id": 10, "name": "traffic light", "supercategory": "outdoor"},
    {"id": 13, "name": "stop sign", "supercategory": "outdoor"},
    {"id": 81, "name": "auto_rickshaw", "supercategory": "vehicle"},  # Indian traffic custom category
]

df_categories = pd.DataFrame(coco_traffic_categories)
print("=== COCO INDIAN TRAFFIC TARGET CATEGORIES ===")
display(df_categories)


In [ ]:
# Cell 3: Bounding Box Geometry Transformation Functions
def coco_to_xyxy(bbox, img_w=1280, img_h=720, normalize=False):
    """
    Converts COCO format [x, y, width, height] to [x1, y1, x2, y2].
    If normalize=True, outputs [0.0, 1.0] range.
    """
    x, y, w, h = bbox
    x1, y1 = x, y
    x2, y2 = x + w, y + h
    if normalize:
        return x1 / img_w, y1 / img_h, x2 / img_w, y2 / img_h
    return int(x1), int(y1), int(x2), int(y2)

# Verification test
test_box = [150.0, 200.0, 300.0, 180.0]
x1, y1, x2, y2 = coco_to_xyxy(test_box, normalize=False)
print(f"COCO Format [x, y, w, h]:  {test_box}")
print(f"XYXY Format [x1, y1, x2, y2]: [{x1}, {y1}, {x2}, {y2}]")


In [ ]:
# Cell 4: Real Indian Traffic COCO Dataset Extraction & Annotation Builder
from google.colab import files
from ultralytics import YOLO
import cv2
import numpy as np
import json

print('Upload an Indian traffic image to extract COCO annotations (or cancel for sample scene):')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    img_bytes = uploaded[fname]
    nparr = np.frombuffer(img_bytes, np.uint8)
    image = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
else:
    print('Using sample Indian traffic intersection scene...')
    !wget -q -O sample_coco.jpg https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/bus.jpg
    image = cv2.imread('sample_coco.jpg')
    if image is None:
        image = np.full((720, 1280, 3), (60, 60, 60), dtype=np.uint8)

img_h, img_w = image.shape[:2]
detector = YOLO('yolov8n.pt')
results = detector(image, conf=0.25, verbose=False)[0]

# Build official COCO JSON dataset structure
coco_dataset = {
    'info': {'description': 'Indian Traffic COCO Dataset', 'year': 2026, 'version': '1.0'},
    'images': [{'id': 101, 'file_name': 'traffic_mg_road.jpg', 'width': img_w, 'height': img_h}],
    'annotations': [],
    'categories': [
        {'id': 1, 'name': 'person', 'supercategory': 'traffic'},
        {'id': 3, 'name': 'car', 'supercategory': 'traffic'},
        {'id': 4, 'name': 'motorcycle', 'supercategory': 'traffic'},
        {'id': 5, 'name': 'auto_rickshaw', 'supercategory': 'traffic'},
        {'id': 6, 'name': 'bus', 'supercategory': 'traffic'},
        {'id': 8, 'name': 'truck', 'supercategory': 'traffic'},
    ]
}

ann_id = 1
for box in results.boxes:
    cls_id = int(box.cls[0].item())
    raw_name = detector.names[cls_id]
    if raw_name not in ['person', 'car', 'motorcycle', 'bus', 'truck', 'bicycle']:
        continue

    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    bw = x2 - x1
    bh = y2 - y1
    aspect = bw / max(1, bh)

    cat_id = 3  # default car
    cat_name = raw_name
    if raw_name == 'person':
        cat_id = 1
    elif raw_name in ['motorcycle', 'bicycle']:
        cat_id = 4
        cat_name = 'motorcycle'
    elif raw_name == 'bus':
        cat_id = 6
    elif raw_name == 'truck':
        cat_id = 8
    elif raw_name == 'car':
        if 0.65 <= aspect <= 1.25 and 1500 < (bw * bh) < 45000:
            cat_id = 5
            cat_name = 'auto_rickshaw'
        else:
            cat_id = 3

    coco_dataset['annotations'].append({
        'id': ann_id,
        'image_id': 101,
        'category_id': cat_id,
        'category_name': cat_name,
        'bbox': [x1, y1, bw, bh],  # Standard COCO: [x, y, width, height]
        'area': float(bw * bh),
        'iscrowd': 0
    })
    ann_id += 1

print(f'Successfully built COCO annotation set with {len(coco_dataset["annotations"])} annotations across {len(coco_dataset["categories"])} classes.')
print(json.dumps(coco_dataset['annotations'][:3], indent=2))


In [ ]:
# Cell 5: Real Ground-Truth Indian Traffic Visualizer
import matplotlib.pyplot as plt
import pandas as pd

vis_img = image.copy()
cat_map = {c['id']: c['name'] for c in coco_dataset['categories']}

CAT_COLORS = {
    'person': (230, 25, 75),        # Red
    'car': (245, 130, 49),          # Orange
    'motorcycle': (0, 240, 255),    # Cyan
    'auto_rickshaw': (0, 230, 70),  # Green
    'bus': (255, 50, 180),          # Magenta
    'truck': (255, 180, 0),         # Amber
}

records = []
for ann in coco_dataset['annotations']:
    cat_name = ann['category_name']
    x, y, w, h = ann['bbox']
    color = CAT_COLORS.get(cat_name, (0, 255, 0))

    # Convert COCO bbox [x, y, w, h] to xyxy [x1, y1, x2, y2]
    x1, y1, x2, y2 = coco_to_xyxy([x, y, w, h], img_w, img_h, normalize=False)

    # Draw Bounding Box
    cv2.rectangle(vis_img, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
    badge = f"{cat_name.upper()} (Area: {int(ann['area'])})"
    (tw, th), _ = cv2.getTextSize(badge, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
    cv2.rectangle(vis_img, (int(x1), max(0, int(y1) - th - 4)), (int(x1) + tw + 4, int(y1)), color, -1)
    cv2.putText(vis_img, badge, (int(x1) + 2, max(th, int(y1) - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (0, 0, 0), 1, cv2.LINE_AA)

    records.append({'ID': ann['id'], 'Category': cat_name, 'COCO_bbox [x,y,w,h]': ann['bbox'], 'Area (px²)': int(ann['area'])})

plt.figure(figsize=(15, 9))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title(f'COCO Ground-Truth Indian Traffic Annotations Visualizer | Total Annotations: {len(records)}', fontsize=13, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

df_ann = pd.DataFrame(records)
print('=== SAMPLE COCO ANNOTATION RECORDS ===')
print(df_ann.head(10).to_string(index=False))


In [ ]:
# Cell 6: Dataset Audit & Multi-Dimensional Distribution Analysis
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
ann_df = pd.DataFrame(coco_dataset['annotations'])

if not ann_df.empty:
    ann_df['w'] = ann_df['bbox'].apply(lambda b: b[2])
    ann_df['h'] = ann_df['bbox'].apply(lambda b: b[3])
    ann_df['aspect_ratio'] = ann_df['w'] / ann_df['h']

    # COCO scale categorization: small (< 32^2), medium (32^2 to 96^2), large (> 96^2)
    def categorize_scale(area):
        if area < 32**2:
            return 'Small (<32²)'
        elif area <= 96**2:
            return 'Medium (32²-96²)'
        return 'Large (>96²)'

    ann_df['scale_category'] = ann_df['area'].apply(categorize_scale)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # 1. Class Distribution
    sns.countplot(data=ann_df, x='category_name', ax=axes[0], palette='Set2')
    axes[0].set_title('Indian Traffic Category Distribution', fontweight='bold')
    axes[0].set_xlabel('Category', fontweight='bold')
    axes[0].set_ylabel('Object Count', fontweight='bold')
    axes[0].tick_params(axis='x', rotation=25)

    # 2. Object Scale Distribution
    sns.countplot(data=ann_df, x='scale_category', ax=axes[1], order=['Small (<32²)', 'Medium (32²-96²)', 'Large (>96²)'], palette='Blues_d')
    axes[1].set_title('COCO Scale Breakdown (Small/Med/Large)', fontweight='bold')
    axes[1].set_xlabel('Object Scale Category', fontweight='bold')
    axes[1].set_ylabel('Count', fontweight='bold')

    # 3. Aspect Ratio Distribution
    sns.histplot(ann_df['aspect_ratio'], bins=12, kde=True, ax=axes[2], color='#0284c7')
    axes[2].set_title('Bounding Box Aspect Ratio (w / h)', fontweight='bold')
    axes[2].set_xlabel('Aspect Ratio (Width / Height)', fontweight='bold')
    axes[2].set_ylabel('Frequency', fontweight='bold')
    axes[2].axvline(1.0, color='red', linestyle='--', label='Square (1.0)')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

    print('=== COCO DATASET AUDIT SUMMARY ===')
    print(f'Total Validated Traffic Objects: {len(ann_df)}')
    print('Class Counts:')
    print(ann_df['category_name'].value_counts())
    print('\nScale Distribution:')
    print(ann_df['scale_category'].value_counts(normalize=True) * 100)
